In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# DLGenAI Project — Milestone 3
## Smart MCQ Solver Challenge — Retrieval-Augmented Generation (RAG)

**This notebook builds a knowledge base + FAISS index, explores bi-encoder vs. cross-encoder retrieval,
and demonstrates the value (and danger) of RAG context using zero-shot classification.**

**This milestone is designed to familiarize you with the core mechanics of Retrieval-Augmented Generation (RAG). You will learn how to query a vector index for relevant context and use a Cross-Encoder to re-rank the results for maximum accuracy. You will then run side-by-side A/B tests to compare a model's zero-shot inference with and without this context. Finally, we will explore the physical limits of context windows (tuning the correct number of chunks to retrieve) and demonstrate the catastrophic dangers of feeding an LLM incorrect data.**

## Setup — Build Knowledge Base and FAISS Index

Run this cell first. It installs FAISS, loads the models, and builds the knowledge base + index used by every question below.

In [5]:
!pip install faiss-cpu -q

In [4]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
sample_submission = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [6]:
print("Creating knowledge base...")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print(f"Knowledge base size: {len(kb)} documents")

Creating knowledge base...
Knowledge base size: 2000 documents


In [7]:
print("Loading embedding model and creating FAISS index...")
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)

index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base and FAISS index successfully created")
print(f"Embedding dimension: {kb_embeddings.shape[1]}")
print(f"Total vectors indexed: {index.ntotal}")

Loading embedding model and creating FAISS index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base and FAISS index successfully created
Embedding dimension: 384
Total vectors indexed: 2000


## Question 1 — Zero-Shot Classification Baseline

Run `facebook/bart-large-mnli` as a zero-shot classifier on the prompt at row index 150,
using the 5 options (A–E) as candidate labels. Report the predicted probability assigned
to the ground-truth correct option.

In [8]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
correct_letter_150 = row_150['answer']
ans_150 = str(row_150[correct_letter_150])

result_150 = zs(prompt_150, candidate_labels=labels_150)

# Map each label back to its probability score
label_to_score = dict(zip(result_150['labels'], result_150['scores']))
q1_score = round(label_to_score[ans_150], 3)

print(f"Correct option ({correct_letter_150}): {ans_150[:80]}...")
print(f"Predicted probability for correct option: {q1_score}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Correct option (C): The butterfly effect is the phenomenon that a small change in the initial condit...
Predicted probability for correct option: 0.384


## Question 2 — Bi-Encoder (FAISS) Retrieval Rank

Embed the prompt at row index 150 with `all-MiniLM-L6-v2`, query the FAISS index for the
top k=10 most similar documents, and find the exact rank at which the true document
(originally at KB index 150) appears.

In [9]:
k = 10
query_embedding_150 = model.encode([prompt_150])
distances, retrieved_indices = index.search(query_embedding_150, k)

retrieved_indices_150 = retrieved_indices[0]

# Rank (1-indexed) of the true document (KB index 150)
if 150 in retrieved_indices_150:
    q2_rank = int(np.where(retrieved_indices_150 == 150)[0][0]) + 1
else:
    q2_rank = None  # not found in top-k

print(f"Retrieved indices (top {k}): {retrieved_indices_150}")
print(f"Rank of true document (KB index 150): {q2_rank}")

Retrieved indices (top 10): [ 663 1701 1269 1532  576  847 1693 1906  168  150]
Rank of true document (KB index 150): 10


**Concept: The Two-Stage Pipeline (Reranking & Cross-Encoders)
In Question 2, you saw that our FAISS database did not put the true document at rank #1. Why? Because FAISS uses Bi-encoder. 
A Bi-encoder embeds the question and the document separately and just compares the distance (Cosine Similarity). They are fast and allows you to search millions of documents but often miss semantic context.**

**To fix this we use a 2 stage pipeline:**

**1. Retrieval: Use a bi-encoder with FAISS to quickly get the top k possible chunks/documents.**

**2. Reranking: Use a Cross-Encoder to deeply evaluate those top candidates and sort them based on highest semantic similarity.**

**A Cross-Encoder passes the Question and the Document into the Transformer network at the exact same time. The Attention mechanism can directly compare the words in the question to the words in the document, resulting in a highly accurate relevance score.**

**For this we part the prompt and the context and ask it to predict a score.**

**Cross Encoders (Hugging Face)**

## Question 3 — Cross-Encoder Reranking

Take the 10 documents retrieved by FAISS above, score each one against the prompt using
`cross-encoder/ms-marco-MiniLM-L-6-v2`, sort by score, and find the rank of the true
correct document after reranking.

In [10]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

docs_10 = [kb[i] for i in retrieved_indices_150]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

# Sort candidate KB indices by cross-encoder score, highest first
ranked_order = np.argsort(ce_scores)[::-1]
ranked_kb_indices = retrieved_indices_150[ranked_order]

q3_rank = int(np.where(ranked_kb_indices == 150)[0][0]) + 1

print(f"Cross-encoder scores: {ce_scores}")
print(f"Reranked KB indices: {ranked_kb_indices}")
print(f"Rank of true document after reranking: {q3_rank}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder scores: [4.6602316 4.6602316 4.7375226 4.7375226 4.6870446 4.7525764 4.7525764
 4.7525764 4.7072477 4.758512 ]
Reranked KB indices: [ 150 1906  847 1693 1269 1532  168  576 1701  663]
Rank of true document after reranking: 1


## Question 4 — Token Count of a RAG Prompt

Retrieve the top k=5 documents for the prompt at row index 42, concatenate them with a
single space, build the RAG string `"Context: [docs] Question: [prompt]"`, and count the
tokens using the `bert-base-uncased` tokenizer (no truncation).

In [11]:
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])

query_embedding_42 = model.encode([prompt_42])
_, retrieved_indices_42 = index.search(query_embedding_42, 5)

docs_5_42 = [kb[i] for i in retrieved_indices_42[0]]
concatenated_docs_42 = " ".join(docs_5_42)

rag_string_42 = f"Context: {concatenated_docs_42} Question: {prompt_42}"

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tokens_42 = tokenizer(rag_string_42, truncation=False)['input_ids']
q4_token_count = len(tokens_42)

print(f"Total tokens: {q4_token_count}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Total tokens: 216


## Question 5 — Zero-Shot Classification With Ground-Truth Context (Oracle RAG)

Build the RAG string using the *true* document for row index 150 as context, and re-run
the zero-shot classifier from Question 1 to see how the probability of the correct option
changes.

In [12]:
true_doc_150 = kb[150]
rag_string_150 = f"Context: {true_doc_150} Question: {prompt_150}"

result_150_rag = zs(rag_string_150, candidate_labels=labels_150)
label_to_score_rag = dict(zip(result_150_rag['labels'], result_150_rag['scores']))
q5_score = round(label_to_score_rag[ans_150], 3)

print(f"New predicted probability for correct option: {q5_score}")

New predicted probability for correct option: 0.989


**Concept: The Danger of Bad Retrieval (Adversarial RAG) 
The golden rule of Retrieval-Augmented Generation is “Garbage In, Garbage Out.” An LLM places immense trust in the external context you inject into its prompt. If your vector database performs poorly and retrieves an irrelevant or incorrect document, the model will often abandon its own internal reasoning and confidently generate the wrong answer based on that bad data. We do the reranking and constricting the number of chunks that we give to the model for the same reason.**

## Question 6 — Adversarial RAG (Bad Retrieval)

Force the context for row index 150 to be an unrelated document (KB index 999) and re-run
the zero-shot classifier to see how a wrong retrieval can hurt the correct option's
probability.

In [13]:
adversarial_doc = kb[999]
adversarial_string_150 = f"Context: {adversarial_doc} Question: {prompt_150}"

result_150_adv = zs(adversarial_string_150, candidate_labels=labels_150)
label_to_score_adv = dict(zip(result_150_adv['labels'], result_150_adv['scores']))
q6_score = round(label_to_score_adv[ans_150], 3)

print(f"Probability of correct option with adversarial context: {q6_score}")

Probability of correct option with adversarial context: 0.529


## Question 7 — Retrieval Hit Rate

For the first 100 rows of `train.csv`, retrieve the top k=5 documents for each prompt.
A "Hit" occurs when the exact string of the row's correct option appears among the 5
retrieved documents. Report the Hit Rate percentage.

In [14]:
k = 5
hits = 0
n_rows = 100

prompts_100 = train['prompt'].iloc[:n_rows].astype(str).tolist()
query_embeddings_100 = model.encode(prompts_100, show_progress_bar=False)
_, retrieved_indices_100 = index.search(query_embeddings_100, k)

for i in range(n_rows):
    row = train.iloc[i]
    correct_answer_text = str(row[row['answer']])
    retrieved_docs = [kb[idx] for idx in retrieved_indices_100[i]]
    if correct_answer_text in retrieved_docs:
        hits += 1

q7_hit_rate = round((hits / n_rows) * 100, 1)
print(f"Hits: {hits}/{n_rows}")
print(f"Hit Rate: {q7_hit_rate}%")

Hits: 73/100
Hit Rate: 73.0%


## Question 8 — Full RAG Pipeline: Retrieve → Rerank → Augment → Predict → MAP@3

For the first 20 rows of `train.csv`, run the complete pipeline:

1. **Retrieve** top k=5 documents from FAISS.
2. **Rerank** with the cross-encoder and keep the single best document.
3. **Augment** the prompt into a RAG string.
4. **Predict** with the zero-shot classifier over the 5 options.
5. **Score** using MAP@3 on the top-3 ranked options.

In [15]:
def average_precision_at_3(correct_letter, ranked_letters):
    top3 = ranked_letters[:3]
    if correct_letter not in top3:
        return 0.0
    rank = top3.index(correct_letter) + 1
    return 1.0 / rank

n_rows_q8 = 20
ap_scores = []

for i in range(n_rows_q8):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    letters = ['A', 'B', 'C', 'D', 'E']
    labels_i = [str(row[l]) for l in letters]
    correct_letter_i = row['answer']

    # Retrieve
    query_embedding_i = model.encode([prompt_i])
    _, retrieved_indices_i = index.search(query_embedding_i, 5)
    docs_5_i = [kb[idx] for idx in retrieved_indices_i[0]]

    # Rerank -> best single document
    pairs_i = [[prompt_i, doc] for doc in docs_5_i]
    ce_scores_i = cross_encoder.predict(pairs_i)
    best_doc_i = docs_5_i[int(np.argmax(ce_scores_i))]

    # Augment
    rag_string_i = f"Context: {best_doc_i} Question: {prompt_i}"

    # Predict
    result_i = zs(rag_string_i, candidate_labels=labels_i)

    # Map returned labels back to A-E letters, sorted by probability (highest first)
    label_to_letter = dict(zip(labels_i, letters))
    ranked_letters_i = [label_to_letter[lbl] for lbl in result_i['labels']]

    ap_scores.append(average_precision_at_3(correct_letter_i, ranked_letters_i))

q8_map3 = round(np.mean(ap_scores), 3)
print(f"MAP@3 across {n_rows_q8} rows: {q8_map3}")

MAP@3 across 20 rows: 0.975


# Milestone 3 Summary

| Q | Topic | Answer |
|---|---|---|
| Q1 | Zero-Shot Classification Baseline (`bart-large-mnli`) | 0.384 |
| Q2 | FAISS Retrieval Rank (row 150, k=10) | 10 |
| Q3 | Cross-Encoder Reranking (`ms-marco-MiniLM-L-6-v2`) | 1 |
| Q4 | Token Count of RAG Prompt (row 42, top-5 docs) | 216 |
| Q5 | Zero-Shot with Ground-Truth Context (Oracle RAG) | 0.989 |
| Q6 | Adversarial RAG (Wrong Context, KB Index 999) | 0.529 |
| Q7 | Retrieval Hit Rate (First 100 Rows, k=5) | **73.0%** |
| Q8 | Full RAG Pipeline MAP@3 (First 20 Rows) | **0.975** |

### Models Used
- **Bi-Encoder:** `all-MiniLM-L6-v2`
- **Vector Store:** FAISS (`IndexFlatL2`)
- **Cross-Encoder:** `cross-encoder/ms-marco-MiniLM-L-6-v2`
- **Zero-Shot Classifier:** `facebook/bart-large-mnli`

### Pipeline Flow
Knowledge Base → Embeddings → FAISS Retrieval → Cross-Encoder Reranking → RAG Prompt Construction → Zero-Shot Prediction → MAP@3 Evaluation

### Key Results
- Retrieval Hit Rate (Top-5): **73.0%**
- End-to-End RAG MAP@3: **0.975**
- Oracle RAG improves prediction confidence.
- Adversarial retrieval significantly reduces confidence.
- Cross-Encoder reranking helps select the most relevant context before prediction.